# 1. Data exploration: MMRDR colour fundus photographs

Reads the outputs of `python -m drxai.prepare`: the manifest, the integrity report and the duplicate audit.
Nothing here alters the data or the splits. Run `prepare` first.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from drxai import GRADE_NAMES
from drxai.config import REPO_ROOT, cache_dir, load_config, resolve

# Read-only: this notebook loads files written by the pipeline commands and computes nothing that is reported elsewhere.
cfg = load_config(os.environ.get("DRXAI_CONFIG", REPO_ROOT / "configs" / "default.yaml"))
CACHE = cache_dir(cfg)
RUNS = resolve(cfg["paths"]["runs_dir"])

In [ ]:
man = pd.read_csv(CACHE / "manifest.csv", dtype={"image_id": str, "group": str})
report = json.loads((CACHE / "integrity_report.json").read_text())
print(f"{len(man)} images | split source: {report['split_source']}")
print("column mapping:", report["column_mapping"])

## Integrity checks
Every check `prepare` ran. A failed critical check stops the pipeline, so all of these passed or are warnings.

In [ ]:
checks = pd.DataFrame(report["checks"]).T
checks["status"] = np.where(checks["passed"], "PASS", np.where(checks["critical"], "FAIL", "WARN"))
checks[["status", "detail"]]

## Grade distribution and class imbalance
The imbalance is why the sampler weights by the inverse square root of class frequency and why per-class recall is reported.

In [ ]:
table = pd.crosstab(man["grade"], man["split"]).reindex(columns=["train", "val", "test"], fill_value=0)
table.index = [f"{g} {GRADE_NAMES[g]}" for g in table.index]
display(table)
ax = (table / table.sum()).plot.bar(figsize=(7, 3), rot=0, color=["#4B2FB5", "#0B5E4F", "#B5450A"])
ax.set_ylabel("Share of split"); ax.set_title("Grade distribution by split"); plt.tight_layout(); plt.show()

## Example images per grade (training split)
The first two training images of each grade by id, after cropping. Chosen by rule, not by eye.

In [ ]:
tr = man[man["split"] == "train"].sort_values("image_id")
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for g in range(5):
    rows = tr[tr["grade"] == g].head(2)
    for r in range(2):
        a = axes[r, g]; a.axis("off")
        if r < len(rows):
            a.imshow(plt.imread(CACHE / rows.iloc[r]["cache_path"]))
            a.set_title(f"{g} {GRADE_NAMES[g]}" if r == 0 else rows.iloc[r]["image_id"], fontsize=9)
plt.tight_layout(); plt.show()

## Image quality
Sharpness (variance of the Laplacian, green channel), brightness and contrast inside the field of view. The test-set analysis reports performance within sharpness bands whose edges come from these training images.

In [ ]:
qcols = [c for c in ["q_sharpness", "q_brightness", "q_contrast"] if c in man.columns]
fig, axes = plt.subplots(1, len(qcols), figsize=(4.2 * len(qcols), 3))
for a, c in zip(np.atleast_1d(axes), qcols):
    for g in range(5):
        a.hist(man.loc[man.grade == g, c].dropna(), bins=30, histtype="step", label=str(g), density=True)
    a.set_title(c.replace("q_", "")); a.legend(title="grade", fontsize=7)
plt.tight_layout(); plt.show()
man.groupby("grade")[qcols].median().round(2)

In [ ]:
low = pd.read_csv(CACHE / "lowest_sharpness_2pct.csv", dtype={"image_id": str})
print(f"{len(low)} least-sharp images (bottom 2%); grade mix:", low["grade"].value_counts().sort_index().to_dict())
show = low.sort_values("q_sharpness").head(5)
fig, axes = plt.subplots(1, len(show), figsize=(3 * max(len(show), 1), 3), squeeze=False)
for a, (_, r) in zip(axes[0], show.iterrows()):
    a.imshow(plt.imread(CACHE / f"images/{r.image_id}.png")); a.axis("off"); a.set_title(f"{r.image_id}\ngrade {r.grade}", fontsize=8)
plt.tight_layout(); plt.show()

## Duplicate audit
The same photograph stored under two ids, found by correlating high-pass fingerprints. Training images that duplicate a test image are removed; the test set is never changed. Pairs whose grades disagree are label noise.

In [ ]:
dup = pd.read_csv(CACHE / "near_duplicates.csv", dtype={"id_a": str, "id_b": str})
print(f"{len(dup)} duplicate pairs; {int((dup.grade_a != dup.grade_b).sum())} with different grades; "
      f"{report.get('n_train_dropped_as_test_duplicates', 0)} training images dropped")
dup.sort_values("corr", ascending=False).head(10)

## Limitation
No patient identifiers are available, so different photographs of the same eye cannot be kept on one side of a split. The duplicate audit catches copies of a photograph, not repeat photographs.